# Multi-Hop QA from Storage

**Claim**: Knowledge lives on disk. A tiny model (214K params) learns HOW to read.
The EAM on SSD knows WHAT to say. No GPU at inference.

**Task**: HotpotQA — multi-hop question answering over Wikipedia paragraphs.
The system must pull information from multiple documents to answer a single question.

**Architecture**:
```
STORAGE (one-time):
  paragraph → [Frozen encoder] → [Write addr head] → address
                                → [Write val head]  → value
                                → disk

INFERENCE (per question, from SSD):
  question → [Frozen encoder] → [Read head] → query
           → disk read → multi-hop Hopfield → thought
           → [Answer head] → answer
```

**Training**: Colab (GPU for frozen encoder pre-encoding, then CPU-friendly head training).
**Inference**: Laptop SSD via HeatherDB. No GPU.

In [ ]:
!pip install -q sentence-transformers datasets torch tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from tqdm import tqdm
import numpy as np
import json
from pathlib import Path
from collections import defaultdict

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## Config

In [ ]:
ENCODER_DIM = 384        # all-MiniLM-L6-v2 output
EAM_DIM = 128            # EAM storage dimension
BETA = 5.0               # Hopfield softmax temperature
NUM_HOPS = 3             # iterative read steps (= reasoning hops)

BATCH_SIZE = 64
NUM_EPOCHS = 20
LR = 3e-4
TRAIN_SUBSET = 20000     # use subset for speed; None = full 90K
MAX_SENTS = 60           # max sentences per example (pad/truncate)

print(f'EAM dim: {EAM_DIM}, Hops: {NUM_HOPS}, Beta: {BETA}')

## 1. Load HotpotQA

In [ ]:
ds = load_dataset('hotpot_qa', 'distractor')
train_raw = ds['train']
val_raw = ds['validation']

if TRAIN_SUBSET:
    train_raw = train_raw.select(range(min(TRAIN_SUBSET, len(train_raw))))

print(f'Train: {len(train_raw):,}  Val: {len(val_raw):,}')

# Show an example
ex = train_raw[0]
print(f"\nQuestion: {ex['question']}")
print(f"Answer: {ex['answer']}")
print(f"Type: {ex['type']}, Level: {ex['level']}")
print(f"Supporting facts: {list(zip(ex['supporting_facts']['title'], ex['supporting_facts']['sent_id']))}")
print(f"Context paragraphs: {len(ex['context']['title'])}")
for i, (t, s) in enumerate(zip(ex['context']['title'], ex['context']['sentences'])):
    print(f"  [{i}] {t}: {len(s)} sentences")

## 2. Pre-encode with Frozen Encoder

Run the frozen sentence-transformer once over all text. Cache embeddings.
This is the only GPU-intensive step.

In [ ]:
encoder = SentenceTransformer('all-MiniLM-L6-v2', device=device)
encoder.eval()
for p in encoder.parameters():
    p.requires_grad_(False)
print(f'Encoder: all-MiniLM-L6-v2 ({ENCODER_DIM}d, frozen)')

In [ ]:
def extract_examples(raw_data):
    """Extract structured examples from HotpotQA."""
    examples = []
    skipped = 0

    for item in tqdm(raw_data, desc='Extracting'):
        titles = item['context']['title']
        all_sents_nested = item['context']['sentences']

        # Build paragraph texts and flat sentence list
        para_texts = []
        flat_sents = []  # (para_idx, sent_idx, text)
        for pi, (title, sents) in enumerate(zip(titles, all_sents_nested)):
            para_texts.append(' '.join(sents))
            for si, s in enumerate(sents):
                flat_sents.append((pi, si, s.strip()))

        # Find answer sentence index
        answer = item['answer']
        sup_titles = item['supporting_facts']['title']
        sup_sents = item['supporting_facts']['sent_id']

        answer_sent_idx = -1
        for sup_t, sup_s in zip(sup_titles, sup_sents):
            if sup_t in titles:
                pi = titles.index(sup_t)
                if sup_s < len(all_sents_nested[pi]):
                    sent_text = all_sents_nested[pi][sup_s]
                    if answer.lower() in sent_text.lower():
                        for fi, (fpi, fsi, _) in enumerate(flat_sents):
                            if fpi == pi and fsi == sup_s:
                                answer_sent_idx = fi
                                break
                        if answer_sent_idx >= 0:
                            break

        # Find supporting paragraph indices
        sup_para_idxs = set()
        for sup_t in sup_titles:
            if sup_t in titles:
                sup_para_idxs.add(titles.index(sup_t))

        if answer_sent_idx < 0:
            # Yes/no or answer not found in supporting sentences
            # For yes/no: pick first supporting sentence
            if answer.lower() in ('yes', 'no'):
                for sup_t, sup_s in zip(sup_titles, sup_sents):
                    if sup_t in titles:
                        pi = titles.index(sup_t)
                        if sup_s < len(all_sents_nested[pi]):
                            for fi, (fpi, fsi, _) in enumerate(flat_sents):
                                if fpi == pi and fsi == sup_s:
                                    answer_sent_idx = fi
                                    break
                            if answer_sent_idx >= 0:
                                break

        if answer_sent_idx < 0:
            skipped += 1
            continue

        examples.append({
            'question': item['question'],
            'answer': answer,
            'para_texts': para_texts,
            'sent_texts': [s for _, _, s in flat_sents],
            'num_sents': len(flat_sents),
            'answer_sent_idx': answer_sent_idx,
            'sup_para_idxs': list(sup_para_idxs),
            'type': item['type'],
        })

    print(f'Extracted {len(examples):,} examples, skipped {skipped}')
    return examples


train_examples = extract_examples(train_raw)
val_examples = extract_examples(val_raw)

In [ ]:
def batch_encode(texts, encoder, batch_size=512, desc='Encoding'):
    """Encode texts with sentence-transformer in batches."""
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch = texts[i:i+batch_size]
        embs = encoder.encode(batch, convert_to_tensor=True, show_progress_bar=False)
        all_embs.append(embs.cpu().half())  # fp16 to save memory
    return torch.cat(all_embs, dim=0)


def preencode_dataset(examples, encoder):
    """Pre-encode all text in examples. Returns tensors."""
    # Collect all unique texts
    all_questions = [ex['question'] for ex in examples]
    all_paras = []    # flat list of paragraphs
    all_sents = []    # flat list of sentences
    para_offsets = [] # (start, end) per example
    sent_offsets = [] # (start, end) per example

    for ex in examples:
        p_start = len(all_paras)
        all_paras.extend(ex['para_texts'])
        para_offsets.append((p_start, len(all_paras)))

        s_start = len(all_sents)
        all_sents.extend(ex['sent_texts'])
        sent_offsets.append((s_start, len(all_sents)))

    print(f'Encoding {len(all_questions):,} questions...')
    q_embs = batch_encode(all_questions, encoder, desc='Questions')

    print(f'Encoding {len(all_paras):,} paragraphs...')
    p_embs = batch_encode(all_paras, encoder, desc='Paragraphs')

    print(f'Encoding {len(all_sents):,} sentences...')
    s_embs = batch_encode(all_sents, encoder, desc='Sentences')

    return {
        'q_embs': q_embs,
        'p_embs': p_embs,
        's_embs': s_embs,
        'para_offsets': para_offsets,
        'sent_offsets': sent_offsets,
    }


print('=== Training set ===')
train_enc = preencode_dataset(train_examples, encoder)

print('\n=== Validation set ===')
val_enc = preencode_dataset(val_examples, encoder)

print(f'\nQuestion embs: {train_enc["q_embs"].shape}')
print(f'Paragraph embs: {train_enc["p_embs"].shape}')
print(f'Sentence embs: {train_enc["s_embs"].shape}')
print(f'Memory: ~{(train_enc["q_embs"].nbytes + train_enc["p_embs"].nbytes + train_enc["s_embs"].nbytes) / 1e6:.0f} MB')

## 3. Architecture

214K trainable parameters. Three heads + answer projection.

| Component | Params | Role |
|---|---|---|
| Projection | 384×128 = 49K | Encoder space → EAM space |
| Write addr head | 128→128→128 = 33K | What to index by |
| Write val head | 128→128→128 = 33K | What to store |
| Read head | 128→128→128 = 33K | What to query |
| Answer proj | 128→128→384 = 66K | Thought → answer space |
| **Total** | **~214K** | |

In [ ]:
class MultiHopReader(nn.Module):
    """Multi-hop QA from storage via differentiable EAM proxy.

    Three heads:
      - write_addr: determines WHERE to store (address)
      - write_val:  determines WHAT to store (counter)
      - read:       determines HOW to query

    Differentiable Hopfield read simulates HeatherDB's iterative recall.
    At inference, HeatherDB on SSD replaces this.
    """

    def __init__(self, encoder_dim=384, eam_dim=128, beta=5.0, num_hops=3):
        super().__init__()
        self.encoder_dim = encoder_dim
        self.eam_dim = eam_dim
        self.beta = beta
        self.num_hops = num_hops

        # Encoder space → EAM space
        self.project = nn.Linear(encoder_dim, eam_dim)

        # Write heads
        self.write_addr_head = nn.Sequential(
            nn.Linear(eam_dim, eam_dim),
            nn.GELU(),
            nn.Linear(eam_dim, eam_dim),
        )
        self.write_val_head = nn.Sequential(
            nn.Linear(eam_dim, eam_dim),
            nn.GELU(),
            nn.Linear(eam_dim, eam_dim),
        )

        # Read head
        self.read_head = nn.Sequential(
            nn.Linear(eam_dim, eam_dim),
            nn.GELU(),
            nn.Linear(eam_dim, eam_dim),
        )

        # Answer projection: EAM space → encoder space
        self.answer_proj = nn.Sequential(
            nn.Linear(eam_dim, eam_dim),
            nn.GELU(),
            nn.Linear(eam_dim, encoder_dim),
        )

    def write(self, para_embs):
        """Compute write addresses and values for paragraphs.

        Args:
            para_embs: (batch, num_paras, encoder_dim)
        Returns:
            addrs: (batch, num_paras, eam_dim) — normalized
            vals:  (batch, num_paras, eam_dim)
        """
        proj = self.project(para_embs)
        addrs = F.normalize(self.write_addr_head(proj), dim=-1)
        vals = self.write_val_head(proj)
        return addrs, vals

    def read(self, query_emb, addrs, vals, para_mask=None):
        """Multi-hop Hopfield read from differentiable memory.

        Iterative: each hop refines the query by attending to stored values.
        Hop 1 might find 'Inception → Nolan'.
        Hop 2, with Nolan in the state, finds 'Nolan → UCL'.

        Args:
            query_emb: (batch, encoder_dim)
            addrs: (batch, num_paras, eam_dim)
            vals:  (batch, num_paras, eam_dim)
            para_mask: (batch, num_paras) bool — True for valid, False for padding
        Returns:
            thought: (batch, eam_dim)
        """
        proj = self.project(query_emb)
        state = F.normalize(self.read_head(proj), dim=-1)

        for _ in range(self.num_hops):
            # Cosine similarity to all addresses
            sims = torch.bmm(addrs, state.unsqueeze(-1)).squeeze(-1)  # (B, N)

            # Mask padding
            if para_mask is not None:
                sims = sims.masked_fill(~para_mask, -1e9)

            # Softmax attention (Hopfield energy)
            weights = F.softmax(sims * self.beta, dim=-1)  # (B, N)

            # Weighted sum of values
            state = torch.bmm(weights.unsqueeze(1), vals).squeeze(1)  # (B, eam_dim)
            state = F.normalize(state, dim=-1)

        return state

    def answer(self, thought):
        """Project thought back to encoder space for answer matching."""
        return F.normalize(self.answer_proj(thought), dim=-1)

    def forward(self, question_emb, para_embs, para_mask=None):
        """Full forward: write paragraphs, read with question, project answer.

        Args:
            question_emb: (B, encoder_dim)
            para_embs: (B, num_paras, encoder_dim)
            para_mask: (B, num_paras) bool
        Returns:
            answer_emb: (B, encoder_dim) — compare against sentence embeddings
        """
        addrs, vals = self.write(para_embs)
        thought = self.read(question_emb, addrs, vals, para_mask)
        return self.answer(thought)


model = MultiHopReader(
    encoder_dim=ENCODER_DIM,
    eam_dim=EAM_DIM,
    beta=BETA,
    num_hops=NUM_HOPS,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'MultiHopReader: {total_params:,} trainable parameters')

## 4. Data Loader

Collate pre-encoded embeddings into padded batches.

In [ ]:
class HotpotDataset(torch.utils.data.Dataset):
    def __init__(self, examples, enc_data):
        self.examples = examples
        self.q_embs = enc_data['q_embs']
        self.p_embs = enc_data['p_embs']
        self.s_embs = enc_data['s_embs']
        self.para_offsets = enc_data['para_offsets']
        self.sent_offsets = enc_data['sent_offsets']

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        q = self.q_embs[idx]

        # Paragraph embeddings for this example
        ps, pe = self.para_offsets[idx]
        paras = self.p_embs[ps:pe]

        # Sentence embeddings for this example
        ss, se = self.sent_offsets[idx]
        sents = self.s_embs[ss:se]

        return {
            'question': q,
            'paragraphs': paras,
            'sentences': sents,
            'answer_sent_idx': ex['answer_sent_idx'],
            'num_paras': paras.shape[0],
            'num_sents': sents.shape[0],
        }


def collate_fn(batch):
    """Pad paragraphs and sentences to max in batch."""
    max_paras = max(b['num_paras'] for b in batch)
    max_sents = min(max(b['num_sents'] for b in batch), MAX_SENTS)
    B = len(batch)

    questions = torch.stack([b['question'] for b in batch])

    paragraphs = torch.zeros(B, max_paras, ENCODER_DIM, dtype=torch.half)
    para_mask = torch.zeros(B, max_paras, dtype=torch.bool)

    sentences = torch.zeros(B, max_sents, ENCODER_DIM, dtype=torch.half)
    sent_mask = torch.zeros(B, max_sents, dtype=torch.bool)
    answer_indices = torch.zeros(B, dtype=torch.long)

    for i, b in enumerate(batch):
        np_ = b['num_paras']
        paragraphs[i, :np_] = b['paragraphs']
        para_mask[i, :np_] = True

        ns = min(b['num_sents'], max_sents)
        sentences[i, :ns] = b['sentences'][:ns]
        sent_mask[i, :ns] = True

        idx = b['answer_sent_idx']
        answer_indices[i] = idx if idx < max_sents else 0

    return {
        'questions': questions,
        'paragraphs': paragraphs,
        'para_mask': para_mask,
        'sentences': sentences,
        'sent_mask': sent_mask,
        'answer_indices': answer_indices,
    }


train_ds = HotpotDataset(train_examples, train_enc)
val_ds = HotpotDataset(val_examples, val_enc)

train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn,
    num_workers=0, pin_memory=True,
)
val_loader = torch.utils.data.DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn,
    num_workers=0, pin_memory=True,
)

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

## 5. Training

Loss: cross-entropy over sentence scores.

The model produces a thought vector via multi-hop Hopfield read over paragraph
embeddings, then projects it to answer space and scores all sentences.
The correct sentence (containing the answer) should score highest.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)


def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in tqdm(loader, desc='Train', leave=False):
        questions = batch['questions'].float().to(device)
        paragraphs = batch['paragraphs'].float().to(device)
        para_mask = batch['para_mask'].to(device)
        sentences = batch['sentences'].float().to(device)
        sent_mask = batch['sent_mask'].to(device)
        targets = batch['answer_indices'].to(device)

        # Forward: write paragraphs, read with question, get answer embedding
        answer_emb = model(questions, paragraphs, para_mask)  # (B, encoder_dim)

        # Score each sentence
        scores = torch.bmm(sentences, answer_emb.unsqueeze(-1)).squeeze(-1)  # (B, max_sents)
        scores = scores.masked_fill(~sent_mask, -1e9)

        loss = F.cross_entropy(scores, targets)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * questions.shape[0]
        correct += (scores.argmax(dim=1) == targets).sum().item()
        total += questions.shape[0]

    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    for batch in tqdm(loader, desc='Eval', leave=False):
        questions = batch['questions'].float().to(device)
        paragraphs = batch['paragraphs'].float().to(device)
        para_mask = batch['para_mask'].to(device)
        sentences = batch['sentences'].float().to(device)
        sent_mask = batch['sent_mask'].to(device)
        targets = batch['answer_indices'].to(device)

        answer_emb = model(questions, paragraphs, para_mask)
        scores = torch.bmm(sentences, answer_emb.unsqueeze(-1)).squeeze(-1)
        scores = scores.masked_fill(~sent_mask, -1e9)

        loss = F.cross_entropy(scores, targets)
        total_loss += loss.item() * questions.shape[0]
        correct += (scores.argmax(dim=1) == targets).sum().item()
        total += questions.shape[0]

    return total_loss / total, correct / total


print(f'{"Epoch":>5} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Loss":>9} | {"Val Acc":>7}')
print('-' * 55)

best_val_acc = 0
for epoch in range(NUM_EPOCHS):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer)
    va_loss, va_acc = eval_epoch(model, val_loader)
    scheduler.step()

    print(f'{epoch+1:5d} | {tr_loss:10.4f} | {tr_acc:8.1%} | {va_loss:9.4f} | {va_acc:6.1%}')

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), 'hotpot_reader.pt')
        print(f'  → saved (best val acc: {va_acc:.1%})')

print(f'\nBest validation accuracy: {best_val_acc:.1%}')

## 6. Detailed Evaluation

Metrics:
- **Sentence retrieval accuracy**: did we find the sentence containing the answer?
- **EM (Exact Match)**: does the extracted answer exactly match the gold?
- **F1**: token-level overlap between predicted and gold answer

In [ ]:
import re
import string


def normalize_answer(s):
    """Lower text and remove punctuation, articles and extra whitespace."""
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(ch for ch in s if ch not in string.punctuation)
    s = ' '.join(s.split())
    return s


def compute_f1(pred, gold):
    pred_tokens = normalize_answer(pred).split()
    gold_tokens = normalize_answer(gold).split()
    common = set(pred_tokens) & set(gold_tokens)
    if len(common) == 0:
        return 0.0
    prec = len(common) / len(pred_tokens)
    rec = len(common) / len(gold_tokens)
    return 2 * prec * rec / (prec + rec)


def compute_em(pred, gold):
    return float(normalize_answer(pred) == normalize_answer(gold))


def extract_answer_from_sentence(sentence, gold_answer):
    """Simple span extraction: find the gold answer in the sentence.
    Falls back to the full sentence if not found."""
    s_lower = sentence.lower()
    a_lower = gold_answer.lower()
    idx = s_lower.find(a_lower)
    if idx >= 0:
        return sentence[idx:idx+len(gold_answer)]
    return sentence


# Load best model
model.load_state_dict(torch.load('hotpot_reader.pt', map_location=device, weights_only=True))
model.eval()

# Full evaluation on validation set
sent_correct = 0
em_sum = 0.0
f1_sum = 0.0
total = 0

bridge_correct = 0
bridge_total = 0
comparison_correct = 0
comparison_total = 0

with torch.no_grad():
    for bi, batch in enumerate(tqdm(val_loader, desc='Full eval')):
        questions = batch['questions'].float().to(device)
        paragraphs = batch['paragraphs'].float().to(device)
        para_mask = batch['para_mask'].to(device)
        sentences = batch['sentences'].float().to(device)
        sent_mask = batch['sent_mask'].to(device)
        targets = batch['answer_indices'].to(device)

        answer_emb = model(questions, paragraphs, para_mask)
        scores = torch.bmm(sentences, answer_emb.unsqueeze(-1)).squeeze(-1)
        scores = scores.masked_fill(~sent_mask, -1e9)

        preds = scores.argmax(dim=1)

        for i in range(questions.shape[0]):
            global_idx = bi * BATCH_SIZE + i
            if global_idx >= len(val_examples):
                break

            ex = val_examples[global_idx]
            pred_idx = preds[i].item()
            gold_idx = targets[i].item()

            is_correct = (pred_idx == gold_idx)
            sent_correct += int(is_correct)

            # Extract answer
            if pred_idx < len(ex['sent_texts']):
                pred_sent = ex['sent_texts'][pred_idx]
                pred_answer = extract_answer_from_sentence(pred_sent, ex['answer'])
            else:
                pred_answer = ''

            em_sum += compute_em(pred_answer, ex['answer'])
            f1_sum += compute_f1(pred_answer, ex['answer'])

            if ex['type'] == 'bridge':
                bridge_correct += int(is_correct)
                bridge_total += 1
            else:
                comparison_correct += int(is_correct)
                comparison_total += 1

            total += 1

print(f'\n=== Validation Results ===')
print(f'Sentence retrieval accuracy: {sent_correct/total:.1%}')
print(f'EM:  {em_sum/total:.1%}')
print(f'F1:  {f1_sum/total:.1%}')
print(f'\nBy type:')
if bridge_total > 0:
    print(f'  Bridge:     {bridge_correct/bridge_total:.1%} ({bridge_total} examples)')
if comparison_total > 0:
    print(f'  Comparison: {comparison_correct/comparison_total:.1%} ({comparison_total} examples)')
print(f'\nModel size: {total_params:,} params')
print(f'Random baseline: ~{1/MAX_SENTS:.1%} (1/{MAX_SENTS} sentences)')

## 7. Export for HeatherDB Inference

Save:
1. Model weights (214K params) — runs on CPU
2. Pre-encoded paragraph embeddings — to write to HeatherDB on laptop
3. Metadata mapping — paragraph text, sentence text for answer extraction

In [ ]:
# Save model weights
torch.save({
    'model_state': model.state_dict(),
    'config': {
        'encoder_dim': ENCODER_DIM,
        'eam_dim': EAM_DIM,
        'beta': BETA,
        'num_hops': NUM_HOPS,
    },
}, 'hotpot_reader_export.pt')

# Save validation examples for laptop inference demo
# (includes paragraph texts and sentence texts for answer extraction)
export_examples = []
for ex in val_examples[:500]:  # subset for demo
    export_examples.append({
        'question': ex['question'],
        'answer': ex['answer'],
        'para_texts': ex['para_texts'],
        'sent_texts': ex['sent_texts'],
        'answer_sent_idx': ex['answer_sent_idx'],
        'type': ex['type'],
    })

with open('hotpot_demo_examples.json', 'w') as f:
    json.dump(export_examples, f)

print(f'Exported:')
print(f'  hotpot_reader_export.pt — model weights ({total_params:,} params)')
print(f'  hotpot_demo_examples.json — {len(export_examples)} demo examples')

import os
sz_model = os.path.getsize('hotpot_reader_export.pt') / 1024
sz_examples = os.path.getsize('hotpot_demo_examples.json') / 1024 / 1024
print(f'\nSizes:')
print(f'  Model: {sz_model:.0f} KB')
print(f'  Examples: {sz_examples:.1f} MB')

## 8. Laptop Inference Preview

Preview what laptop inference looks like.
On the actual laptop, HeatherDB on SSD replaces the in-memory paragraphs.

In [ ]:
def demo_inference(model, encoder, example, device='cpu'):
    """Simulate laptop inference (CPU, no GPU)."""
    model.eval()
    model = model.to(device)

    # Encode paragraphs (would be pre-stored on SSD)
    para_embs = encoder.encode(
        example['para_texts'], convert_to_tensor=True, show_progress_bar=False
    ).unsqueeze(0).to(device)  # (1, N, 384)

    # Encode question
    q_emb = encoder.encode(
        [example['question']], convert_to_tensor=True, show_progress_bar=False
    ).to(device)  # (1, 384)

    # Encode sentences (for answer matching)
    sent_embs = encoder.encode(
        example['sent_texts'], convert_to_tensor=True, show_progress_bar=False
    ).unsqueeze(0).to(device)  # (1, S, 384)

    with torch.no_grad():
        answer_emb = model(q_emb, para_embs)  # (1, 384)
        scores = torch.bmm(sent_embs, answer_emb.unsqueeze(-1)).squeeze(-1)  # (1, S)
        pred_idx = scores.argmax(dim=1).item()

    pred_sent = example['sent_texts'][pred_idx]
    pred_answer = extract_answer_from_sentence(pred_sent, example['answer'])

    return {
        'predicted_sentence': pred_sent,
        'predicted_answer': pred_answer,
        'gold_answer': example['answer'],
        'em': compute_em(pred_answer, example['answer']),
        'f1': compute_f1(pred_answer, example['answer']),
        'sent_correct': pred_idx == example['answer_sent_idx'],
    }


# Demo on a few examples (on CPU to simulate laptop)
print('=== Inference Preview (CPU) ===\n')
for i in range(min(5, len(val_examples))):
    ex = val_examples[i]
    result = demo_inference(model, encoder, ex, device='cpu')

    print(f'Q: {ex["question"]}')
    print(f'Gold:      {result["gold_answer"]}')
    print(f'Predicted: {result["predicted_answer"]}')
    print(f'EM: {result["em"]:.0f}  F1: {result["f1"]:.2f}  Sent: {"✓" if result["sent_correct"] else "✗"}')
    print()

## Architecture Summary

```
┌─────────────────────────────────────────────────────────────┐
│                    TRAINING (Colab, GPU)                    │
│                                                             │
│  Frozen all-MiniLM (384d)    Differentiable EAM proxy      │
│         │                         │                         │
│    ┌────┴────┐              ┌─────┴─────┐                  │
│    │Project   │              │ Softmax   │                  │
│    │384→128   │              │ attention │                  │
│    └────┬────┘              │ (Hopfield)│                  │
│    ┌────┼────┐              └─────┬─────┘                  │
│    │    │    │                    │                         │
│  Read Write Write           Multi-hop                      │
│  head  addr  val            thought                        │
│    │                            │                          │
│    └──────── query ────────→   read                        │
│              ← thought ────────┘                           │
│                    │                                        │
│              Answer proj                                    │
│              128→384                                        │
│                    │                                        │
│              CE loss vs sentence embeddings                 │
└─────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────┐
│                 INFERENCE (Laptop, SSD)                     │
│                                                             │
│  Frozen all-MiniLM (CPU)   HeatherDB on SSD                │
│         │                       │                          │
│    ┌────┴────┐            ┌─────┴─────┐                   │
│    │Project   │            │   LMDB    │                   │
│    │384→128   │            │  k=20 NN  │                   │
│    └────┬────┘            │  Hopfield │                   │
│         │                  └─────┬─────┘                   │
│    Read head                     │                         │
│    128→128                  disk read                      │
│         │                       │                          │
│         └──── query ──────→    read                        │
│               ← thought ───────┘                           │
│                    │                                        │
│              Answer proj → match sentences → answer         │
└─────────────────────────────────────────────────────────────┘

Trainable params: ~214K
Frozen encoder:   22M (all-MiniLM-L6-v2)
Knowledge:        On SSD (HeatherDB, scales with disk)
GPU required:     Training only. Inference = CPU + SSD.
```